In [32]:
!nvidia-smi

Sat May 23 14:07:13 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [33]:
!pip install -q google-genai pdf2image pillow pandas fuzzywuzzy python-Levenshtein
!pip install fuzzywuzzy python-Levenshtein
!pip install pdfplumber
import json
import os
import pandas as pd
from fuzzywuzzy import fuzz
from itertools import product


In [34]:
from google.colab import drive
drive.mount('/content/drive')

# 🔧 Change this to your actual folder path
DATA_DIR = '/content/drive/MyDrive/magriplast'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [35]:
# Mise à jour du chemin vers le dossier identifié dans le Drive
DATA_DIR = '/content/drive/MyDrive/magriplast'

In [36]:
!ls /content/drive/MyDrive

'Colab Notebooks'
 magriplast
 ph5_colect.zip
'RAPPORT D'\''AUDIT TECHNIQUE : SÉCURITÉ RÉSEAU ET PÉRIMÈTRE (PHASE 2).gdoc'


In [37]:
!ls -F /content/drive/MyDrive/magriplast

20260302_12105033.pdf  20260302_12181839.pdf  20260302_12260491.pdf
20260302_12115701.pdf  20260302_12185159.pdf


In [38]:

import os
import json
import time
from getpass import getpass

from google import genai
from google.genai import types
from pdf2image import convert_from_path
import pandas as pd
from fuzzywuzzy import fuzz

# --- Gemini API key ---
# You can paste it directly (less secure) or enter interactively
GEMINI_API_KEY = getpass("Enter your Gemini API key: ")

client = genai.Client(api_key=GEMINI_API_KEY)
model_name = "gemini-2.5-flash"  # or gemini-2.5-pro if needed

Enter your Gemini API key: ··········


In [39]:
def extract_page_lines(image, page_num):
    """
    Sends a page image to Gemini and returns a list of line items.
    Each item is a dict with keys:
      ref, designation, qty, prix_unitaire, doc_type
    """
    prompt = """You are an OCR and data extraction assistant for French/Tunisian business documents.
Look at the page image and return ALL product line items in strict JSON format.

For each line item, include:
  - "ref": the article/product reference code exactly as printed (can be null if not visible)
  - "designation": the product description
  - "qty": quantity (number)
  - "prix_unitaire": unit price in TND (number, 3 decimal places)
  - "doc_type": "BC" if the page is a purchase order / bon de commande,
                "FACTURE" if it is an invoice,
                "UNKNOWN" if unclear

Return ONLY a JSON array of objects, nothing else.
Example: [{"ref":"P123456","designation":"BUTEE A RLX","qty":4,"prix_unitaire":122.341,"doc_type":"BC"}]
"""

    try:
        response = client.models.generate_content(
            model=model_name,
            contents=[prompt, image],
            config=types.GenerateContentConfig(
                temperature=0.0,
                max_output_tokens=8192,
                response_mime_type="application/json",
            ),
        )
        raw_text = response.text.strip()
        # Parse JSON
        data = json.loads(raw_text)
        if isinstance(data, list):
            for item in data:
                item.setdefault("ref", None)
                item.setdefault("designation", "")
                item.setdefault("qty", None)
                item.setdefault("prix_unitaire", None)
                item.setdefault("doc_type", "UNKNOWN")
            return data
        else:
            print(f"Page {page_num}: unexpected JSON structure")
            return []
    except Exception as e:
        print(f"Page {page_num}: Gemini error - {e}")
        return []

In [40]:
all_lines = []

pdf_files = [f for f in os.listdir(DATA_DIR) if f.lower().endswith('.pdf')]
print(f"Found {len(pdf_files)} PDF files")

for idx, pdf_name in enumerate(pdf_files):
    pdf_path = os.path.join(DATA_DIR, pdf_name)
    print(f"\n[{idx+1}/{len(pdf_files)}] Processing: {pdf_name}")

    try:
        # Convert PDF pages to images (300 DPI for readability)
        images = convert_from_path(pdf_path, dpi=300)
    except Exception as e:
        print(f"  ❌ Could not convert PDF: {e}")
        continue

    for page_num, img in enumerate(images):
        print(f"  Page {page_num+1}/{len(images)} ...", end="")
        lines = extract_page_lines(img, page_num+1)
        if lines:
            for item in lines:
                item["source_pdf"] = pdf_name
                item["page"] = page_num + 1
            all_lines.extend(lines)
            print(f" {len(lines)} lines extracted")
        else:
            print(" no lines found")
        # Small delay to respect rate limits (adjust as needed)
        time.sleep(1)

print(f"\n✅ Extraction complete. Total lines collected: {len(all_lines)}")

Found 5 PDF files

[1/5] Processing: 20260302_12115701.pdf
  Page 1/3 ... 9 lines extracted
  Page 2/3 ... 12 lines extracted
  Page 3/3 ... no lines found

[2/5] Processing: 20260302_12185159.pdf
  Page 1/6 ... 1 lines extracted
  Page 2/6 ... 2 lines extracted
  Page 3/6 ... 1 lines extracted
  Page 4/6 ... 2 lines extracted
  Page 5/6 ... 2 lines extracted
  Page 6/6 ... 4 lines extracted

[3/5] Processing: 20260302_12105033.pdf
  Page 1/8 ... 20 lines extracted
  Page 2/8 ... 20 lines extracted
  Page 3/8 ... no lines found
  Page 4/8 ... 20 lines extracted
  Page 5/8 ... 5 lines extracted
  Page 6/8 ... 6 lines extracted
  Page 7/8 ... 6 lines extracted
  Page 8/8 ... 5 lines extracted

[4/5] Processing: 20260302_12181839.pdf
  Page 1/9 ... 11 lines extracted
  Page 2/9 ... 5 lines extracted
  Page 3/9 ... 5 lines extracted
  Page 4/9 ... 2 lines extracted
  Page 5/9 ... 2 lines extracted
  Page 6/9 ... 4 lines extracted
  Page 7/9 ... 4 lines extracted
  Page 8/9 ... 4 lines extr

In [41]:
df = pd.DataFrame(all_lines)
print(f"Columns: {list(df.columns)}")
print(f"Doc types found: {df['doc_type'].value_counts().to_dict()}")
df.head()

Columns: ['ref', 'designation', 'qty', 'prix_unitaire', 'doc_type', 'source_pdf', 'page']
Doc types found: {'UNKNOWN': 65, 'FACTURE': 54, 'BC': 49}


,ref,designation,qty,prix_unitaire,doc_type,source_pdf,page
0,2019005,MW UNION = TECHNOPOLYMERE,10.0,5.568,FACTURE,20260302_12115701.pdf,1
1,NPC12-02,SHAKO RACCORD DROIT 12 1/4,10.0,6.207,FACTURE,20260302_12115701.pdf,1
2,NPC10-04,SHAKO RACC DROIT 10 1/2 EQ 2L01,5.0,7.752,FACTURE,20260302_12115701.pdf,1
3,NPC8-04,SHAKO RACCORD DROIT 8 1/2,5.0,6.821,FACTURE,20260302_12115701.pdf,1
4,PU1080BL,CAM TUBE 10X8 BLEU,20.0,4.339,FACTURE,20260302_12115701.pdf,1


In [56]:
# --- Pair BC ↔ Facture per PDF (tolerant) ---
alias_pairs = []

for pdf_name, group in df.groupby('source_pdf'):
    bc_lines = group[group['doc_type'] == 'BC']
    fac_lines = group[group['doc_type'] == 'FACTURE']
    if bc_lines.empty or fac_lines.empty:
        continue

    for idx_bc, bc_row in bc_lines.iterrows():
        best_score = 0
        best_fac = None
        for idx_fac, fac_row in fac_lines.iterrows():
            # Quantity match with small tolerance
            if (bc_row['qty'] is None or fac_row['qty'] is None or
                abs(float(bc_row['qty']) - float(fac_row['qty'])) > 0.5):
                continue
            # Price match with small tolerance (TND)
            if (bc_row['prix_unitaire'] is None or fac_row['prix_unitaire'] is None or
                abs(float(bc_row['prix_unitaire']) - float(fac_row['prix_unitaire'])) > 0.02):
                continue

            # Fuzzy designation score (still useful for confidence)
            score = fuzz.token_sort_ratio(
                str(bc_row['designation']).lower(),
                str(fac_row['designation']).lower()
            )
            if score > best_score:
                best_score = score
                best_fac = fac_row

        # Lower threshold to 50, also accept pairs with exact qty/price even if low score
        if best_fac is not None and (best_score >= 50):
            alias_pairs.append({
                'pdf': pdf_name,
                'internal_ref': bc_row['ref'],
                'external_ref': best_fac['ref'],
                'designation': bc_row['designation'],
                'qty': bc_row['qty'],
                'prix': bc_row['prix_unitaire'],
                'match_score': best_score
            })

aliases_df = pd.DataFrame(alias_pairs)
print(f"Candidate alias pairs: {len(aliases_df)}")
# Keep only where refs differ
aliases_df = aliases_df[aliases_df['internal_ref'] != aliases_df['external_ref']]
print(f"After removing identical refs: {len(aliases_df)}")
# Deduplicate
aliases_df = aliases_df.sort_values('match_score', ascending=False)
aliases_df = aliases_df.drop_duplicates(subset=['internal_ref', 'external_ref'], keep='first')
aliases_df.reset_index(drop=True, inplace=True)
aliases_df.head(20)
# --- Add known aliases that might still be missing ---
verified = [
    {'pdf':'20260302_12115701.pdf', 'internal_ref':'P199420414', 'external_ref':'81105 TN',
     'designation':'BUTEE A RLX CYLIND', 'qty':4.0, 'prix':122.341, 'match_score':100},
    {'pdf':'20260302_12115701.pdf', 'internal_ref':'P199420416', 'external_ref':'NKX 25 Z',
     'designation':'RLT A AIG AVEC BUTEE', 'qty':4.0, 'prix':242.200, 'match_score':100},
    {'pdf':'20260302_12115701.pdf', 'internal_ref':'P199684214', 'external_ref':'2019005',
     'designation':'SHAKO UNION TECHNOPOLYMERE', 'qty':10.0, 'prix':5.568, 'match_score':100},
]
verified_df = pd.DataFrame(verified)
aliases_df = pd.concat([aliases_df, verified_df], ignore_index=True)
aliases_df = aliases_df.drop_duplicates(subset=['internal_ref', 'external_ref'], keep='first')
aliases_df = aliases_df.sort_values('match_score', ascending=False).reset_index(drop=True)
print(f"Total aliases after adding verified: {len(aliases_df)}")

Candidate alias pairs: 37
After removing identical refs: 37
Total aliases after adding verified: 37


In [57]:
# --- Clean external refs: remove spaces, standardize ---
aliases_df['external_ref'] = aliases_df['external_ref'].str.replace(' ', '').str.upper()
aliases_df['internal_ref'] = aliases_df['internal_ref'].str.replace(' ', '').str.upper()

# Remove duplicates again after cleaning
aliases_df = aliases_df.drop_duplicates(subset=['internal_ref', 'external_ref'], keep='first')
aliases_df = aliases_df.sort_values('match_score', ascending=False).reset_index(drop=True)

print(f"Final aliases after cleaning: {len(aliases_df)}")
display(aliases_df[['external_ref', 'internal_ref', 'designation', 'qty', 'prix', 'match_score']])

Final aliases after cleaning: 37


,external_ref,internal_ref,designation,qty,prix,match_score
0,8020137,P199680105,DECAPANT 5L,1.0,30.300,100
1,PU1080BL,P199681847,CAM TUBE 10X8 BLEU,20.0,4.339,100
2,RACCORD,P199600771,RACCORD S16 male 1/2,8.0,12.000,100
3,8012599,P199692133,SILICONE SIKAFLEX CRISTAL,1.0,34.252,100
4,NKX25Z,P199420416,RLT A AIG AVEC BUTEE,4.0,242.200,100
5,COLLIER,P199680428,COLLIER DE SERRAGE A BOULON DN 80/85,4.0,15.000,95
6,6050004,P199680440,RIVET ALLU 4.8*22,250.0,0.098,94
7,NPC8-04,P199450101,SHAKO RACC DROIT 8 1/2,5.0,6.821,94
8,NPC12-02,P199450265,SHAKO RACC DROIT 12 1/4,10.0,6.207,94
9,COLLIER,P199681803,COLLIER DE SERRAGE A BOULON DN 23/35,10.0,4.000,92


In [58]:
print("\n=== Proposed alias mappings (external → internal) ===\n")
for _, row in aliases_df.iterrows():
    print(f"{row['external_ref']}  →  {row['internal_ref']}  "
          f"(score: {row['match_score']}%)  [{row['designation'][:60]}...]")


=== Proposed alias mappings (external → internal) ===

8020137  →  P199680105  (score: 100%)  [DECAPANT 5L...]
PU1080BL  →  P199681847  (score: 100%)  [CAM TUBE 10X8 BLEU...]
RACCORD  →  P199600771  (score: 100%)  [RACCORD S16 male 1/2...]
8012599  →  P199692133  (score: 100%)  [SILICONE SIKAFLEX CRISTAL...]
NKX25Z  →  P199420416  (score: 100%)  [RLT A AIG AVEC BUTEE...]
COLLIER  →  P199680428  (score: 95%)  [COLLIER DE SERRAGE A BOULON DN 80/85...]
6050004  →  P199680440  (score: 94%)  [RIVET ALLU 4.8*22...]
NPC8-04  →  P199450101  (score: 94%)  [SHAKO RACC DROIT 8 1/2...]
NPC12-02  →  P199450265  (score: 94%)  [SHAKO RACC DROIT 12 1/4...]
COLLIER  →  P199681803  (score: 92%)  [COLLIER DE SERRAGE A BOULON DN 23/35...]
BCF04H  →  P199403539  (score: 90%)  [MECHE ACIER RECTIFIE 4MM...]
BCF05H  →  P199403542  (score: 90%)  [MECHE ACIER RECTIFIE 5MM...]
FLEXSM  →  P199600774  (score: 90%)  [FLEX EAU R2AT DN 3/8" LG 1000 F/FC 90°...]
1-947  →  P199680240  (score: 90%)  [JEUX 9 CLE MALE LO

In [59]:
sql_statements = []
for _, row in aliases_df.iterrows():
    # Use the supplier name from the invoice; you may adjust this
    supplier = 'BOUDRANT'  # change if needed
    sql = (
        f"INSERT INTO reference_aliases (supplier_name, external_ref, internal_ref, approved_by, approved_at) "
        f"VALUES ('{supplier}', '{row['external_ref']}', '{row['internal_ref']}', 'colab_gemini', NOW());"
    )
    sql_statements.append(sql)

print("\n-- SQL to seed aliases --\n")
print("\n".join(sql_statements))


-- SQL to seed aliases --

INSERT INTO reference_aliases (supplier_name, external_ref, internal_ref, approved_by, approved_at) VALUES ('BOUDRANT', '8020137', 'P199680105', 'colab_gemini', NOW());
INSERT INTO reference_aliases (supplier_name, external_ref, internal_ref, approved_by, approved_at) VALUES ('BOUDRANT', 'PU1080BL', 'P199681847', 'colab_gemini', NOW());
INSERT INTO reference_aliases (supplier_name, external_ref, internal_ref, approved_by, approved_at) VALUES ('BOUDRANT', 'RACCORD', 'P199600771', 'colab_gemini', NOW());
INSERT INTO reference_aliases (supplier_name, external_ref, internal_ref, approved_by, approved_at) VALUES ('BOUDRANT', '8012599', 'P199692133', 'colab_gemini', NOW());
INSERT INTO reference_aliases (supplier_name, external_ref, internal_ref, approved_by, approved_at) VALUES ('BOUDRANT', 'NKX25Z', 'P199420416', 'colab_gemini', NOW());
INSERT INTO reference_aliases (supplier_name, external_ref, internal_ref, approved_by, approved_at) VALUES ('BOUDRANT', 'COLLIE

In [60]:
aliases_df.to_csv('/content/drive/MyDrive/alias_mappings.csv', index=False)
print("Exported to /content/drive/MyDrive/alias_mappings.csv")

Exported to /content/drive/MyDrive/alias_mappings.csv


In [61]:
sql = ""
for _, row in aliases_df.iterrows():
    sql += f"INSERT INTO reference_aliases (supplier_name, external_ref, internal_ref, approved_by, approved_at) "
    sql += f"VALUES ('BOUDRANT', '{row['external_ref']}', '{row['internal_ref']}', 'auto_colab', NOW());\n"
print(sql)

INSERT INTO reference_aliases (supplier_name, external_ref, internal_ref, approved_by, approved_at) VALUES ('BOUDRANT', '8020137', 'P199680105', 'auto_colab', NOW());
INSERT INTO reference_aliases (supplier_name, external_ref, internal_ref, approved_by, approved_at) VALUES ('BOUDRANT', 'PU1080BL', 'P199681847', 'auto_colab', NOW());
INSERT INTO reference_aliases (supplier_name, external_ref, internal_ref, approved_by, approved_at) VALUES ('BOUDRANT', 'RACCORD', 'P199600771', 'auto_colab', NOW());
INSERT INTO reference_aliases (supplier_name, external_ref, internal_ref, approved_by, approved_at) VALUES ('BOUDRANT', '8012599', 'P199692133', 'auto_colab', NOW());
INSERT INTO reference_aliases (supplier_name, external_ref, internal_ref, approved_by, approved_at) VALUES ('BOUDRANT', 'NKX25Z', 'P199420416', 'auto_colab', NOW());
INSERT INTO reference_aliases (supplier_name, external_ref, internal_ref, approved_by, approved_at) VALUES ('BOUDRANT', 'COLLIER', 'P199680428', 'auto_colab', NOW())

In [62]:
# Display final aliases table
print("Final aliases (external → internal):")
display(aliases_df[['external_ref', 'internal_ref', 'designation', 'qty', 'prix', 'match_score']])

# Download the CSV file to your local machine
from google.colab import files
files.download('/content/drive/MyDrive/alias_mappings.csv')

# Save SQL to a file and download
sql_filename = '/content/alias_inserts.sql'
with open(sql_filename, 'w') as f:
    f.write("\n".join(sql_statements))
files.download(sql_filename)

Final aliases (external → internal):


,external_ref,internal_ref,designation,qty,prix,match_score
0,8020137,P199680105,DECAPANT 5L,1.0,30.300,100
1,PU1080BL,P199681847,CAM TUBE 10X8 BLEU,20.0,4.339,100
2,RACCORD,P199600771,RACCORD S16 male 1/2,8.0,12.000,100
3,8012599,P199692133,SILICONE SIKAFLEX CRISTAL,1.0,34.252,100
4,NKX25Z,P199420416,RLT A AIG AVEC BUTEE,4.0,242.200,100
5,COLLIER,P199680428,COLLIER DE SERRAGE A BOULON DN 80/85,4.0,15.000,95
6,6050004,P199680440,RIVET ALLU 4.8*22,250.0,0.098,94
7,NPC8-04,P199450101,SHAKO RACC DROIT 8 1/2,5.0,6.821,94
8,NPC12-02,P199450265,SHAKO RACC DROIT 12 1/4,10.0,6.207,94
9,COLLIER,P199681803,COLLIER DE SERRAGE A BOULON DN 23/35,10.0,4.000,92


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [63]:
# Debug: inspect extraction from 20260302_12115701.pdf
debug_pdf = '20260302_12115701.pdf'
debug_df = df[df['source_pdf'] == debug_pdf]
print(f"Lines extracted from {debug_pdf}:")
print(f"Doc types: {debug_df['doc_type'].value_counts().to_dict()}")
display(debug_df[['page', 'doc_type', 'ref', 'designation', 'qty', 'prix_unitaire']])

Lines extracted from 20260302_12115701.pdf:
Doc types: {'BC': 12, 'FACTURE': 9}


,page,doc_type,ref,designation,qty,prix_unitaire
0,1,FACTURE,2019005,MW UNION = TECHNOPOLYMERE,10.0,5.568
1,1,FACTURE,NPC12-02,SHAKO RACCORD DROIT 12 1/4,10.0,6.207
2,1,FACTURE,NPC10-04,SHAKO RACC DROIT 10 1/2 EQ 2L01,5.0,7.752
3,1,FACTURE,NPC8-04,SHAKO RACCORD DROIT 8 1/2,5.0,6.821
4,1,FACTURE,PU1080BL,CAM TUBE 10X8 BLEU,20.0,4.339
5,1,FACTURE,PU225S-04-S2,SHAKO ELECTROVANNE NF 2/2 1/2,1.0,240.278
6,1,FACTURE,Z10100,SHAKO TEMPORISATEUR,1.0,109.943
7,1,FACTURE,81105 TN,BUTEE A RLX CYLINDRIQUE,4.0,122.341
8,1,FACTURE,NKX 25 Z,RLT A AIGUILLES AVEC BUTEE A B,4.0,242.200
9,2,BC,P199420414,BUTEE A RLX CYLIND ref 81105TN SPEC: BL/LR25/0...,4.0,122.341
